## YoloV8_based_PPE_Complaince_Detection_System

#####  To count the dataset images and labels

In [1]:
from pathlib import Path

root = Path(r"C:\Tasks\YoloV8_based_PPE_Complaince_Detection_System\Dataset")

for s in ["train","test","valid"]:
    img = list((root/s/"images").glob("*.*"))
    lbs = list((root/s/"labels").glob("*.txt"))

    #print(f"images:",len(img))
    #print(f"labels:",len(lbs))

    print(f"{s}: images = {len(img)}, labels ={len(lbs)}")

print(open(root/"data.yaml").read())

train: images = 3248, labels =3248
test: images = 406, labels =406
valid: images = 406, labels =406
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 5
names: ['helmet', 'no_helmet', 'no_vest', 'person', 'vest']



##### To Count the classes:


In [2]:
import yaml
from collections import Counter

names = yaml.safe_load(open(root / "data.yaml"))["names"]
for split in ["train", "valid", "test"]:
    c = Counter()
    for f in (root / split / "labels").glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                c[names[int(line.split()[0])]] += 1
    print(split, dict(c))

train {'helmet': 4068, 'no_vest': 4912, 'person': 8259, 'no_helmet': 3993, 'vest': 2952}
valid {'helmet': 531, 'person': 1042, 'no_vest': 649, 'vest': 350, 'no_helmet': 523}
test {'helmet': 529, 'vest': 396, 'person': 954, 'no_vest': 513, 'no_helmet': 393}


##### Image and box sizes

In [3]:
from PIL import Image
import numpy as np

sizes = Counter()
for img in (root / "train" / "images").glob("*.*"):
    sizes[Image.open(img).size] += 1
print(sizes.most_common(5))

areas = {n: [] for n in names}
for f in (root / "train" / "labels").glob("*.txt"):
    for line in f.read_text().splitlines():
        if line.strip():
            c, x, y, w, h = line.split()[:5]
            areas[names[int(c)]].append(float(w) * float(h))
for n, a in areas.items():
    print(n, "median box area (% of image):", round(np.median(a) * 100, 2))

[((640, 640), 3248)]
helmet median box area (% of image): 0.48
no_helmet median box area (% of image): 0.31
no_vest median box area (% of image): 1.09
person median box area (% of image): 5.76
vest median box area (% of image): 4.43


##### Dataset validations:

In [4]:
bad_imgs, bad_labels, empty = [], [], 0
for split in ["train", "valid", "test"]:
    for img in (root / split / "images").glob("*.*"):
        try:
            Image.open(img).verify()
        except Exception:
            bad_imgs.append(img.name)
    for f in (root / split / "labels").glob("*.txt"):
        lines = [l for l in f.read_text().splitlines() if l.strip()]
        if not lines:
            empty += 1
        for l in lines:
            p = l.split()
            c, x, y, w, h = int(p[0]), *map(float, p[1:5])
            if len(p) < 5 or c > 4 or not all(0 <= v <= 1 for v in (x, y, w, h)) or w == 0 or h == 0:
                bad_labels.append((split, f.name))

print("corrupt images:", len(bad_imgs))
print("bad label lines:", len(bad_labels))
print("empty label files:", empty)

corrupt images: 0
bad label lines: 0
empty label files: 0


##### Smoke test the yolo model with one epochs:

In [7]:
from ultralytics import YOLO

aug = dict(fliplr=0.5,degrees=10,hsv_v=0.4,hsv_s=0.5,scale=0.5)

model = YOLO("yolov8n.pt")
_=model.train(
    data=r"C:\Tasks\YoloV8_based_PPE_Complaince_Detection_System\Dataset\data.yaml",
    epochs=1,
    batch=10,
    imgsz=640,
    device=0,
    workers=2,
    project=str(root.parent/"runs"),
    name="yolov8nsmoketest",
    **aug

)

Ultralytics 8.4.190  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Tasks\YoloV8_based_PPE_Complaince_Detection_System\Dataset\data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, n